In [5]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import balanced_accuracy_score


crops = pd.read_csv("soil_measures.csv")


X = crops.drop("crop", axis=1)
y = crops["crop"]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=15, stratify=y
)


feature_performance = {}

for feature in ["N", "P", "K", "ph"]:
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("log_reg", LogisticRegression(
            max_iter=1000,
            solver="lbfgs"
        ))
    ])

    pipeline.fit(X_train[[feature]], y_train)
    y_pred = pipeline.predict(X_test[[feature]])
    
    score = balanced_accuracy_score(y_test, y_pred)
    feature_performance[feature] = score
    
    print(f"Balanced Accuracy (only {feature}): {score:.4f}")
#### Best Feature #####
best_feature = max(feature_performance, key=feature_performance.get)

best_predictive_feature = {
    best_feature: feature_performance[best_feature]
}

print("\nBest single predictive feature:")
print(best_predictive_feature)

#### Full Model ###########
full_model = Pipeline([
    ("scaler", StandardScaler()),
    ("log_reg", LogisticRegression(
        max_iter=1000,
        solver="lbfgs"
    ))
])

full_model.fit(X_train, y_train)

y_pred_full = full_model.predict(X_test)
full_feature_score = balanced_accuracy_score(y_test, y_pred_full)

print(f"\nBalanced Accuracy (ALL features): {full_feature_score:.4f}")

Balanced Accuracy (only N): 0.1500
Balanced Accuracy (only P): 0.1955
Balanced Accuracy (only K): 0.2773
Balanced Accuracy (only ph): 0.1045

Best single predictive feature:
{'K': 0.2772727272727273}

Balanced Accuracy (ALL features): 0.6955
